# E2 — decodability of the QuIC embedding (n=14 consumer)

**Question.** Which graph properties are linearly readable from the full
sorted output vector, and in what basis does that content live?

**Answers** (canonical angles, r=1 — config locked here for the paper;
non-canonical arm dropped, since C4 at 0.998 leaves it no headroom):

Linear probe on raw sorted coordinates, 5-fold CV, complete enumeration
of connected cubic graphs on 14 vertices:
tri **1.000**, C4 **0.998**, C5 **0.928 ± 0.092**, girth 0.993 (sanity
row — threshold function of the counts), spectral gap 0.944, diameter
0.548 (informative negative: not a cut-moment quantity, and the probe
correctly finds it only half-accessible).

The moment ladder localizes the content. Triangles are carried by purity
alone: Σp² reads tri at 0.999. C4 is carried exactly by Σp² through Σp⁵,
saturating at k=5 at its full-vector ceiling. C5 is absent from the
first eight power sums (≤0.11) *and* from degree-2 interactions of the
first four (0.14 ± 0.10) — yet reads at 0.93 from the sorted coordinates
directly. Its carrier is therefore finer symmetric-function content of
the probability multiset than low-order moment polynomials: the
L-statistic side of the sorted vector, not the moment side. This is the
functional-form face of the E1 stratification result, where C5 was
readable only within fixed-(tri, C4) cells: conditioning is the only
moment-adjacent route to C5.

Accessibility depth is a step function of cycle length — tri needs one
power sum, C4 four, C5 none of the first eight. The lemma's targets, in
order: purity affine in tr(A³); C4 entering the span at Σp⁵; and the
θ_mix expansion should say *why* the fifth frequency moment is where
4-cycle content becomes separable.

**The journey — kept in full below as the record.** The first two
tables in this notebook are wrong, and they stay: (1) RidgeCV's
closed-form LOO selector fails numerically in the near-interpolating
regime (leverage → 1, the 1/(1−h_ii) term underflows), which crushed the
first pass to R² ≈ 0.08 and survived a grid extension because the grid
was never the problem; diagnosed by manual alpha sweep and fixed with
explicit nested CV (`cv=5`). (2) The first moment-ladder run showed an
impossible non-monotonicity at k=6 — high-k power sums collapse toward a
rank-1 family and the ridge solve destabilizes; fixed by switching that
cell to SVD least squares. Broken outputs are the provenance for the
protocol choices; the n=16 clone bakes the fixes in from the top and
points back here.

**Artifacts of record:** n14_e2_ridge_probe_v3.pkl (results + frozen
splits — E3 must reuse these exact splits), n14_powersum_check_v3.pkl,
n14_moment_ladder.pkl. Earlier-version pkls are superseded.

**Verb discipline:** CV folds partition the complete enumeration —
"linearly decodable," never "predicts" or "generalizes."

# E2 — Ridge probe (n=14 consumer)

Full sorted vectors → RidgeCV per target, KFold(5, shuffle, SEED). Pillar-1 headline
decodability table: R² ± σ per target for {tri, C4, C5, girth, diameter, spectral gap}.

Protocol: alpha selected *inside* each training fold (RidgeCV's internal LOO-GCV on
the train split only — no leakage). The 5 fold splits are frozen and pickled so E3's
GIN probe reuses them verbatim.

Appended: the power-sum check (plan-ordered before E2, shares this split object).

In [4]:
import pickle

import numpy as np
import networkx as nx

from numpy.linalg import eigvalsh
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [5]:
SEED = 0
ALPHAS = np.logspace(-6, 2, 9)
N_SPLITS = 5

In [6]:
base = "/kaggle/input/notebooks/lukemiller1987/n14-aaai-dataset/"
with open(base + "n14_data_dict.pkl", 'rb') as f:
    data_dict = pickle.load(f)

keys = sorted(data_dict)
vectors       = np.vstack([data_dict[k]['exact_vector'] for k in keys])
num_triangles = np.array([data_dict[k]['num_triangles'] for k in keys])
num_4_cycles  = np.array([data_dict[k]['num_4_cycles']  for k in keys])
num_5_cycles  = np.array([data_dict[k]['num_5_cycles']  for k in keys])

## Extra targets

Producer cell 19 added `girth` / `diameter` / `spectral_gap` fields, but README v1
doesn't list them — the shipped pkl may predate that cell. Read the fields if
present, else compute from graph6/adj_mat (509 graphs, seconds either way).

Spectral gap = λ₁ − λ₂ of A. On cubic λ₁ = 3, so this equals the Laplacian
algebraic connectivity (L = 3I − A) — the two definitions coincide here.

In [7]:
def get_or_compute(field, fn):
    if field in data_dict[keys[0]]:
        return np.array([data_dict[k][field] for k in keys], dtype=float)
    return np.array([fn(k) for k in keys], dtype=float)

_graphs = None
def graph(k):
    global _graphs
    if _graphs is None:
        _graphs = {k: nx.from_graph6_bytes(data_dict[k]['graph6'].encode()) for k in keys}
    return _graphs[k]

girth        = get_or_compute('girth',        lambda k: nx.girth(graph(k)))
diameter     = get_or_compute('diameter',     lambda k: nx.diameter(graph(k)))
spectral_gap = get_or_compute('spectral_gap',
                              lambda k: (lambda s: s[-1] - s[-2])(np.sort(eigvalsh(data_dict[k]['adj_mat']))))

print('girth values:   ', sorted(set(girth.astype(int))))
print('diameter values:', sorted(set(diameter.astype(int))))

girth values:    [np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
diameter values: [np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]


## Probe machinery

Full vectors go in raw — coordinates are commensurate (all probabilities), and
z-scoring 16,384 near-constant tail dims against n=509 manufactures noise features.
If a target's R² disappoints, standardization is the first lever to try before
escalating to E5.

In [8]:
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
splits = [(tr, te) for tr, te in kf.split(vectors)]   # frozen; E3 reuses these

def ridge_probe(X, y, splits=splits, alphas=ALPHAS, standardize=False):
    """Per-fold: RidgeCV fit on train (alpha via internal LOO-GCV), R2 on test."""
    scores, chosen = [], []
    for tr, te in splits:
        if standardize:
            model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas))
        else:
            model = RidgeCV(alphas=alphas)
        model.fit(X[tr], y[tr])
        scores.append(r2_score(y[te], model.predict(X[te])))
        a = model[-1].alpha_ if standardize else model.alpha_
        chosen.append(a)
    return np.array(scores), chosen

## E2 table

In [9]:
targets = {
    'triangles':    num_triangles.astype(float),
    '4-cycles':     num_4_cycles.astype(float),
    '5-cycles':     num_5_cycles.astype(float),
    'girth':        girth,
    'diameter':     diameter,
    'spectral_gap': spectral_gap,
}

e2_results = {}
print(f'{"target":>13}   R2 mean +- sd    per-fold R2                          alphas chosen')
for name, y in targets.items():
    sc, al = ridge_probe(vectors, y)
    e2_results[name] = {'r2_folds': sc, 'r2_mean': sc.mean(),
                        'r2_sd': sc.std(), 'alphas_chosen': al}
    folds = ' '.join(f'{s:+.3f}' for s in sc)
    print(f'{name:>13}:  {sc.mean():+.3f} +- {sc.std():.3f}   [{folds}]   {sorted(set(al))}')

       target   R2 mean +- sd    per-fold R2                          alphas chosen
    triangles:  +0.079 +- 0.004   [+0.080 +0.072 +0.079 +0.080 +0.086]   [np.float64(1e-06)]
     4-cycles:  +0.005 +- 0.006   [+0.006 +0.013 +0.010 -0.003 -0.000]   [np.float64(1e-06)]
     5-cycles:  -0.009 +- 0.014   [+0.002 +0.001 +0.005 -0.027 -0.026]   [np.float64(1e-06)]
        girth:  +0.039 +- 0.010   [+0.037 +0.050 +0.039 +0.048 +0.022]   [np.float64(1e-06)]
     diameter:  +0.021 +- 0.009   [+0.030 +0.014 +0.023 +0.030 +0.009]   [np.float64(1e-06)]
 spectral_gap:  +0.043 +- 0.004   [+0.038 +0.048 +0.048 +0.043 +0.039]   [np.float64(1e-06)]


Interpretation guards for the write-up:
- **girth** on connected cubic n=14 is a deterministic threshold function of
  (tri>0, C4>0, C5>0) — its row is a sanity check, not independent evidence.
  Same caveat, weaker, for diameter (range is narrow at n=14).
- Fold σ at n=509 with ~100-graph test folds is honest but wide; the n=16 rerun
  (E4) is where the table tightens.

In [10]:
out = {
    'splits': [(tr.tolist(), te.tolist()) for tr, te in splits],
    'seed': SEED, 'n_splits': N_SPLITS, 'alphas_grid': ALPHAS,
    'results': e2_results,
    'note': 'E3 must reuse these exact splits for the matched-CV comparison.',
}
with open('/kaggle/working/n14_e2_ridge_probe.pkl', 'wb') as f:
    pickle.dump(out, f)
print('saved n14_e2_ridge_probe.pkl')

saved n14_e2_ridge_probe.pkl


## Power-sum check

Three scalars per graph — Σp², Σp³, Σp⁴ — as the *entire* feature set, same splits,
same protocol. If these recover most of the full-vector R², the linearly accessible
content is moment-shaped: explains effective rank ~2, confirms the descriptor story,
and tells the lemma to start at purity.

Power sums are standardized (unlike the full vectors): three non-commensurate
scales, and the alpha grid bottoms out at 1e-6.

In [11]:
P = np.stack([(vectors ** 2).sum(axis=1),
              (vectors ** 3).sum(axis=1),
              (vectors ** 4).sum(axis=1)], axis=1)

psum_results = {}
print(f'{"target":>13}   full-vector R2     power-sum R2 (3 features)')
for name in ['triangles', '4-cycles', '5-cycles']:
    sc, _ = ridge_probe(P, targets[name], standardize=True)
    psum_results[name] = {'r2_folds': sc, 'r2_mean': sc.mean(), 'r2_sd': sc.std()}
    full = e2_results[name]
    print(f'{name:>13}:  {full["r2_mean"]:+.3f} +- {full["r2_sd"]:.3f}     '
          f'{sc.mean():+.3f} +- {sc.std():.3f}')

with open('/kaggle/working/n14_powersum_check.pkl', 'wb') as f:
    pickle.dump({'results': psum_results, 'splits_from': 'n14_e2_ridge_probe.pkl'}, f)
print('\nsaved n14_powersum_check.pkl')

       target   full-vector R2     power-sum R2 (3 features)
    triangles:  +0.079 +- 0.004     +1.000 +- 0.000
     4-cycles:  +0.005 +- 0.006     +0.678 +- 0.040
     5-cycles:  -0.009 +- 0.014     +0.062 +- 0.059

saved n14_powersum_check.pkl


## E2 first pass — read before trusting the table

**Power-sum check is the finding.** Triangles are *exactly* affine in
(Σp², Σp³, Σp⁴): R² = 1.000 ± 0.000 across folds. The lemma's target
relation — purity carrying tr(A³) — is observed as a numerically exact
functional dependence before it's proven. C4 at 0.678 is partial capture,
C5 at 0.062 is essentially absent — precisely the moment-order story:
each cycle length sits deeper in the moment ladder, and three power sums
only reach the top of it. This also explains effective rank ~2 and tells
the lemma to start at purity and stop arguing about it.

**Full-vector rows are void.** Every fold on every target selected the
grid minimum (1e-6): boundary-pinned alpha, classic mis-scaled grid.
Coordinates are probabilities (~1e-4 scale); the centered Gram spectrum
lives far below 1e-6, so the entire grid over-regularizes. R² = 0.079 on
triangles is an artifact. Grid extended to logspace(-14, 2, 17); table
below this cell supersedes the one above.

**Expectation for the rerun, stated pre-data:** tri R² should rise
substantially (PC1 geometry demands it) but may legitimately trail the
power sums — Σp² is *quadratic* in the coordinates, inexpressible by a
coordinate-linear probe. If a gap survives the fix, that is itself a
result: linear-in-moments ⊃ linear-in-coordinates, and it hands E5 its
opening sentence.

In [12]:
ALPHAS = np.logspace(-14, 2, 17)

In [13]:
e2_results = {}
print(f'{"target":>13}   R2 mean +- sd    per-fold R2                          alphas chosen')
for name, y in targets.items():
    sc, al = ridge_probe(vectors, y)
    e2_results[name] = {'r2_folds': sc, 'r2_mean': sc.mean(),
                        'r2_sd': sc.std(), 'alphas_chosen': al}
    folds = ' '.join(f'{s:+.3f}' for s in sc)
    print(f'{name:>13}:  {sc.mean():+.3f} +- {sc.std():.3f}   [{folds}]   {sorted(set(al))}')

       target   R2 mean +- sd    per-fold R2                          alphas chosen
    triangles:  +0.079 +- 0.004   [+0.080 +0.072 +0.079 +0.080 +0.086]   [np.float64(1e-06)]
     4-cycles:  +0.005 +- 0.006   [+0.006 +0.013 +0.010 -0.003 -0.000]   [np.float64(1e-06)]
     5-cycles:  -0.009 +- 0.014   [+0.002 +0.001 +0.005 -0.027 -0.026]   [np.float64(1e-06)]
        girth:  +0.039 +- 0.010   [+0.037 +0.050 +0.039 +0.048 +0.022]   [np.float64(1e-06)]
     diameter:  +0.021 +- 0.009   [+0.030 +0.014 +0.023 +0.030 +0.009]   [np.float64(1e-06)]
 spectral_gap:  +0.043 +- 0.004   [+0.038 +0.048 +0.048 +0.043 +0.039]   [np.float64(1e-06)]


In [15]:
out = {
    'splits': [(tr.tolist(), te.tolist()) for tr, te in splits],
    'seed': SEED, 'n_splits': N_SPLITS, 'alphas_grid': ALPHAS,
    'results': e2_results,
    'note': 'E3 must reuse these exact splits for the matched-CV comparison.',
}
with open('/kaggle/working/n14_e2_ridge_probe_v2.pkl', 'wb') as f:
    pickle.dump(out, f)
print('saved n14_e2_ridge_probe.pkl')

saved n14_e2_ridge_probe.pkl


In [17]:
P = np.stack([(vectors ** 2).sum(axis=1),
              (vectors ** 3).sum(axis=1),
              (vectors ** 4).sum(axis=1)], axis=1)

psum_results = {}
print(f'{"target":>13}   full-vector R2     power-sum R2 (3 features)')
for name in ['triangles', '4-cycles', '5-cycles']:
    sc, _ = ridge_probe(P, targets[name], standardize=True)
    psum_results[name] = {'r2_folds': sc, 'r2_mean': sc.mean(), 'r2_sd': sc.std()}
    full = e2_results[name]
    print(f'{name:>13}:  {full["r2_mean"]:+.3f} +- {full["r2_sd"]:.3f}     '
          f'{sc.mean():+.3f} +- {sc.std():.3f}')

with open('/kaggle/working/n14_powersum_check_v2.pkl', 'wb') as f:
    pickle.dump({'results': psum_results, 'splits_from': 'n14_e2_ridge_probe.pkl'}, f)
print('\nsaved n14_powersum_check_v2.pkl')

       target   full-vector R2     power-sum R2 (3 features)
    triangles:  +0.079 +- 0.004     +1.000 +- 0.000
     4-cycles:  +0.005 +- 0.006     +0.678 +- 0.040
     5-cycles:  -0.009 +- 0.014     +0.062 +- 0.059

saved n14_powersum_check_v2.pkl


In [18]:
# E2 diagnostic: is the linear failure real, or a probe artifact?

# (1) guard against stale kernel state
print('ALPHAS live in kernel:', ALPHAS.min(), '..', ALPHAS.max(), f'({len(ALPHAS)} pts)')

# (2) GCV bypass — manual test-fold R2 across alphas, fold 0
from sklearn.linear_model import Ridge, LinearRegression
tr, te = splits[0]
y = targets['triangles']
print('\nalpha        test R2 (fold 0, triangles)')
for a in np.logspace(-16, 2, 19):
    m = Ridge(alpha=a).fit(vectors[tr], y[tr])
    print(f'{a:8.0e}   {r2_score(y[te], m.predict(vectors[te])):+7.3f}')

# (3) decisive: do the principal axes linearly predict tri AT ALL?
from numpy.linalg import svd
from scipy.stats import spearmanr, pearsonr
Xc = vectors - vectors.mean(axis=0)
U, S, Vt = svd(Xc, full_matrices=False)
pc_scores = Xc @ Vt[:10].T
var = S**2 / (S**2).sum()
print('\nPC    var     tri Pearson   tri Spearman')
for i in range(5):
    pr = pearsonr(pc_scores[:, i], y).statistic
    sr = spearmanr(pc_scores[:, i], y).statistic
    print(f'PC{i+1}   {var[i]:.3f}   {pr:+.3f}        {sr:+.3f}')

# OLS on top-10 PCs, same folds (PCs fit on all data — mild leakage, fine for diagnosis)
sc = np.array([r2_score(y[te], LinearRegression().fit(pc_scores[tr], y[tr])
               .predict(pc_scores[te])) for tr, te in splits])
print(f'\nOLS on top-10 global PCs: R2 = {sc.mean():+.3f} +- {sc.std():.3f}')

ALPHAS live in kernel: 1e-14 .. 100.0 (17 pts)

alpha        test R2 (fold 0, triangles)
   1e-16    +1.000
   1e-15    +1.000
   1e-14    +1.000
   1e-13    +1.000
   1e-12    +1.000
   1e-11    +1.000
   1e-10    +1.000
   1e-09    +0.997
   1e-08    +0.941
   1e-07    +0.497
   1e-06    +0.080
   1e-05    +0.008
   1e-04    +0.001
   1e-03    +0.000
   1e-02    +0.000
   1e-01    -0.000
   1e+00    -0.000
   1e+01    -0.000
   1e+02    -0.000

PC    var     tri Pearson   tri Spearman
PC1   0.512   +0.981        +0.976
PC2   0.153   -0.090        -0.003
PC3   0.078   +0.123        +0.107
PC4   0.077   -0.002        -0.026
PC5   0.053   +0.090        +0.132

OLS on top-10 global PCs: R2 = +0.999 +- 0.000


## E2 protocol note + first results

**Probe fix.** RidgeCV's default closed-form LOO selector fails numerically
in the near-interpolating regime (p ≫ n, tiny alpha): leverage h_ii → 1 and
the 1/(1−h_ii) term underflows, inflating LOO error for exactly the alphas
that generalize best. Diagnosed by manual alpha sweep (test R² = 1.000 at
alpha ≤ 1e-10 vs 0.080 at the GCV choice). Fixed with explicit nested CV
(`RidgeCV(cv=5)`). Chosen alphas at the grid floor are expected and correct:
exact-arithmetic data, noiseless target relation, optimal penalty ≈ 0.

**Triangles are coordinate-linear.** Global PC1 carries tri at Pearson
+0.981; OLS on the top ten PCs reaches R² = 0.999. Together with the
power-sum result (tri exactly affine in Σp², Σp³, Σp⁴; R² = 1.000 ± 0.000),
the leading accessible content is simultaneously low-dimensional, linear,
and moment-shaped — the three views of the same fact the cut-moment lemma
should produce. Purity-first route confirmed empirically.

**C4 and C5 are now the live rows.** Power sums alone: 0.678 and 0.062 —
the moment ladder runs out fast. The fixed full-vector probe measures how
much deeper coordinate-linear reads go; E1's within-stratum PC1 takeover
(C4 at |ρ| ≈ 0.98 once tri is fixed) predicts substantial unconditioned C4
recovery. Gaps that remain are E5's opening, not a defect.

In [21]:
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
splits = [(tr, te) for tr, te in kf.split(vectors)]   # frozen; E3 reuses these

def ridge_probe(X, y, splits=splits, alphas=ALPHAS, standardize=False):
    """Per-fold: RidgeCV fit on train (alpha via internal LOO-GCV), R2 on test."""
    scores, chosen = [], []
    for tr, te in splits:
        if standardize:
            model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5))
        else:
            model = RidgeCV(alphas=alphas, cv=5)
        model.fit(X[tr], y[tr])
        scores.append(r2_score(y[te], model.predict(X[te])))
        a = model[-1].alpha_ if standardize else model.alpha_
        chosen.append(a)
    return np.array(scores), chosen

In [22]:

e2_results = {}
print(f'{"target":>13}   R2 mean +- sd    per-fold R2                          alphas chosen')
for name, y in targets.items():
    sc, al = ridge_probe(vectors, y)
    e2_results[name] = {'r2_folds': sc, 'r2_mean': sc.mean(),
                        'r2_sd': sc.std(), 'alphas_chosen': al}
    folds = ' '.join(f'{s:+.3f}' for s in sc)
    print(f'{name:>13}:  {sc.mean():+.3f} +- {sc.std():.3f}   [{folds}]   {sorted(set(al))}')

       target   R2 mean +- sd    per-fold R2                          alphas chosen
    triangles:  +1.000 +- 0.000   [+1.000 +1.000 +1.000 +1.000 +1.000]   [np.float64(1e-12)]
     4-cycles:  +0.998 +- 0.002   [+0.997 +0.999 +0.996 +0.999 +0.999]   [np.float64(1e-13)]
     5-cycles:  +0.928 +- 0.092   [+0.968 +0.948 +0.746 +0.987 +0.988]   [np.float64(1e-13), np.float64(1e-12)]
        girth:  +0.993 +- 0.014   [+0.965 +1.000 +1.000 +1.000 +1.000]   [np.float64(1e-14)]
     diameter:  +0.548 +- 0.058   [+0.606 +0.501 +0.476 +0.626 +0.532]   [np.float64(1e-12), np.float64(1e-10), np.float64(1e-09)]
 spectral_gap:  +0.944 +- 0.010   [+0.937 +0.937 +0.935 +0.958 +0.955]   [np.float64(1e-13), np.float64(1e-12)]


In [23]:
out = {
    'splits': [(tr.tolist(), te.tolist()) for tr, te in splits],
    'seed': SEED, 'n_splits': N_SPLITS, 'alphas_grid': ALPHAS,
    'results': e2_results,
    'note': 'E3 must reuse these exact splits for the matched-CV comparison.',
}
with open('/kaggle/working/n14_e2_ridge_probe_v3.pkl', 'wb') as f:
    pickle.dump(out, f)
print('saved n14_e2_ridge_probe_v3.pkl')

saved n14_e2_ridge_probe_v3.pkl


In [24]:
P = np.stack([(vectors ** 2).sum(axis=1),
              (vectors ** 3).sum(axis=1),
              (vectors ** 4).sum(axis=1)], axis=1)

psum_results = {}
print(f'{"target":>13}   full-vector R2     power-sum R2 (3 features)')
for name in ['triangles', '4-cycles', '5-cycles']:
    sc, _ = ridge_probe(P, targets[name], standardize=True)
    psum_results[name] = {'r2_folds': sc, 'r2_mean': sc.mean(), 'r2_sd': sc.std()}
    full = e2_results[name]
    print(f'{name:>13}:  {full["r2_mean"]:+.3f} +- {full["r2_sd"]:.3f}     '
          f'{sc.mean():+.3f} +- {sc.std():.3f}')

with open('/kaggle/working/n14_powersum_check_v3.pkl', 'wb') as f:
    pickle.dump({'results': psum_results, 'splits_from': 'n14_e2_ridge_probe.pkl'}, f)
print('\nsaved n14_powersum_check_v3.pkl')

       target   full-vector R2     power-sum R2 (3 features)
    triangles:  +1.000 +- 0.000     +1.000 +- 0.000
     4-cycles:  +0.998 +- 0.002     +0.754 +- 0.043
     5-cycles:  +0.928 +- 0.092     +0.064 +- 0.066

saved n14_powersum_check_v3.pkl


## E2 — headline decodability (n=14, canonical, r=1)

**All three cycle counts are coordinate-linear.** tri R² = 1.000,
C4 = 0.998, C5 = 0.928 ± 0.092, from a linear probe on the raw sorted
vector. Against the power-sum column this is the moment ladder made
empirical:

|                          | tri   | C4    | C5    |
|--------------------------|-------|-------|-------|
| Σp², Σp³, Σp⁴ (3 feats)  | 1.000 | 0.754 | 0.064 |
| full vector, linear      | 1.000 | 0.998 | 0.928 |

Each cycle length sits one rung deeper: triangles are fully carried by
the lowest power sums, C4 only partly, C5 not at all — yet all three
remain within *linear* reach of the sorted coordinates. Lexicographic
accessibility, now in R² form. Lemma directive: purity delivers tri;
C4/C5 must be sought in higher power sums or finer symmetric functions.

**Contrast rows.** girth 0.993 is a sanity check, not evidence — on
connected cubic n=14 it is a threshold function of counts already
decoded. spectral_gap 0.944: spectral content is linearly present,
previewing E2.5. diameter 0.548 is the informative negative: a global
metric quantity, not a cut-moment functional, and the probe correctly
finds it only half-accessible. The table is not trivially saturated —
the probe distinguishes moment-shaped structure from the rest.

**Protocol notes.** Chosen alphas sit at/near the grid floor — correct
behavior for exact-arithmetic, noiseless targets (protocol cell above).
C5's σ is one fold (0.746 vs 0.95–0.99 elsewhere); report as ±σ, likely
a few high-C5 leverage graphs in that split — the n=16 replication
adjudicates. Power-sum C4 reads 0.754 here vs 0.678 pre-fix: the
standardized pipeline shared the broken LOO selector; this table
supersedes n14_powersum_check_v2.pkl. Verb discipline for the paper:
CV folds partition the *complete* enumeration, so this is
in-distribution linear decodability — write "linearly decodable,"
never "predicts" or "generalizes."

In [26]:
# Moment ladder: R2 vs number of power sums, k = 2..K
K = 8
PS = np.stack([(vectors ** k).sum(axis=1) for k in range(2, K + 1)], axis=1)
print(f'{"k up to":>8}   tri      C4       C5')
for j in range(1, K):
    row = [ridge_probe(PS[:, :j], targets[t], standardize=True)[0].mean()
           for t in ['triangles', '4-cycles', '5-cycles']]
    print(f'{j+1:>8}   ' + '  '.join(f'{r:+.3f}' for r in row))

 k up to   tri      C4       C5
       2   +0.999  +0.002  +0.020
       3   +1.000  +0.677  +0.062
       4   +1.000  +0.754  +0.064


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.7482e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=9.79634e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.7482e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=9.79634e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.7482e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.

       5   +1.000  +0.972  +0.065


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.38768e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=8.86467e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=2.70277e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.91847e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.42969e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/pytho

       6   +1.000  +0.592  +0.064


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.05505e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.12671e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.66373e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.02947e-18): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=4.03341e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/pytho

       7   +1.000  +0.989  +0.064


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=5.63041e-18): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.3594e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.4253e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=3.23381e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.2896e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.

       8   +1.000  +0.997  +0.066


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.2896e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


In [27]:
def lstsq_probe(X, y, splits=splits):
    """CV R2 via min-norm least squares — stable under collinear features."""
    scores = []
    for tr, te in splits:
        mu, sd = X[tr].mean(0), X[tr].std(0)
        Xtr = np.column_stack([np.ones(len(tr)), (X[tr] - mu) / sd])
        Xte = np.column_stack([np.ones(len(te)), (X[te] - mu) / sd])
        w, *_ = np.linalg.lstsq(Xtr, y[tr], rcond=None)
        scores.append(r2_score(y[te], Xte @ w))
    return np.array(scores)

## Moment ladder — where each cycle length enters

**Triangles: purity alone.** Σp² by itself reads tri at R² = 0.999 — the
sharpest empirical form of the lemma's target statement. On cubic n=14,
purity is affine in the triangle count, full stop. The lemma's job is now
to derive a relation the data already exhibits exactly.

**C4: one rung down.** Absent at k=2 (0.002), enters with Σp³ (0.68),
saturates at exactly k=5 at 0.997 — the full-vector ceiling — and the
plateau beyond k=5 is flat (updated after the lstsq rerun below). Four
power sums, Σp² through Σp⁵, carry C4 in its entirety: one rung deeper
than tri, with a sharp edge the cut-moment expansion now has to
reproduce.
(Non-monotone k=6 entry in the first run was a numerical artifact of
near-rank-1 high-k power sums; probe swapped to SVD lstsq — see cell.)

**C5: not on this ladder.** Flat at ≤0.11 through eight power sums,
while the full-vector linear probe reads it at 0.93. Its accessible
content is therefore *not* low-order power-sum content — it lives in
finer symmetric functions of the probability multiset. This is the
moment-ladder face of the E1 stratification result: C5 was only readable
*within* fixed-(tri,C4) cells, i.e., conditionally — and conditional
readout corresponds to interactions among moments, not marginal moments.
Two independent analyses, one geometry: the hierarchy is conditional,
and the conditioning depth grows with cycle length. The empirical
grading is exact: tri requires one power sum, C4 requires four, and C5
is absent from the first eight — accessibility depth is a step function
of cycle length.

In [28]:
# Moment ladder: R2 vs number of power sums, k = 2..K
K = 8
PS = np.stack([(vectors ** k).sum(axis=1) for k in range(2, K + 1)], axis=1)
print(f'{"k up to":>8}   tri      C4       C5')
for j in range(1, K):
    row = [lstsq_probe(PS[:, :j], targets[t]).mean()
           for t in ['triangles', '4-cycles', '5-cycles']]
    print(f'{j+1:>8}   ' + '  '.join(f'{r:+.3f}' for r in row))

 k up to   tri      C4       C5
       2   +0.999  +0.001  +0.022
       3   +1.000  +0.677  +0.066
       4   +1.000  +0.755  +0.069
       5   +1.000  +0.997  +0.045
       6   +1.000  +0.997  +0.101
       7   +1.000  +0.997  +0.107
       8   +1.000  +0.997  +0.106


In [29]:
# C5: linear in moment interactions? Degree-2 polynomial over Sum p^2..p^5.
# Decides whether C5's conditional readability (E1) is quadratic-in-moments
# in closed form, or lives in finer symmetric functions than moments.
from sklearn.preprocessing import PolynomialFeatures
Q = PolynomialFeatures(degree=2, include_bias=False).fit_transform(PS[:, :4])
sc = lstsq_probe(Q, targets['5-cycles'])
print(f'C5 ~ deg-2 poly of 4 power sums ({Q.shape[1]} feats): '
      f'R2 = {sc.mean():+.3f} +- {sc.std():.3f}')

C5 ~ deg-2 poly of 4 power sums (14 feats): R2 = +0.143 +- 0.103


In [30]:
# Ladder table from the lstsq run, recomputed and saved — figures build
# from data, not scrollback.
ladder = {}
for j in range(1, K):
    ladder[j + 1] = {t: lstsq_probe(PS[:, :j], targets[t]).mean()
                     for t in ['triangles', '4-cycles', '5-cycles']}
with open('/kaggle/working/n14_moment_ladder.pkl', 'wb') as f:
    pickle.dump({'ladder_r2': ladder, 'probe': 'lstsq_probe (SVD, rcond=None)',
                 'features': 'standardized power sums Sum p^k, k=2..8, nested',
                 'seed': SEED, 'splits_from': 'n14_e2_ridge_probe_v3.pkl'}, f)
print('saved n14_moment_ladder.pkl')

saved n14_moment_ladder.pkl


## E2.5 — spectral decomposition: how much of the embedding is spectrum?

Three questions, per the plan. (a) Regress each top embedding PC on
spectral moments tr(A^k)/n — what share of embedding variance is
spectral-moment content? (b) Find cospectral mates in the enumeration —
any pair has zero spectral distance, so nonzero embedding separation is
super-spectral content *by construction*. (c) The same PC table with
cycle counts and with the full sorted spectrum as competing feature
sets — the "why not spectra" gate, answered with a column instead of an
argument.

Expectations, stated pre-data. On cubic graphs the short-cycle counts
ARE low spectral moments (tr(A²) is constant, tr(A³) = 6·tri exactly,
tr(A⁴) is affine in C4 given regularity, tr(A⁵) in C5 and tri), so PC1
and PC2 should read as almost fully spectral, and the cycles-vs-moments
columns should nearly coincide. The content of the section is the
remainder: variance in the tail PCs that the *full spectrum* cannot
explain, and cospectral separations. Note for the cospectral exhibit:
spectrum plus regularity fixes tri, C4, and C5, so cospectral cubic
mates have identical counted cycles — any embedding separation between
them is beyond the spectrum AND beyond everything this paper has
decoded so far. That is the "plus" in "spectrum plus," quantified
inside the dataset. If no cubic mates exist at n=14, the cell says so
and the exhibit moves to the n=16 clone.

In [31]:
# Spectra: read the producer field if present, else compute.
if 'spectrum' in data_dict[keys[0]]:
    spectra = np.stack([data_dict[k]['spectrum'] for k in keys])
else:
    spectra = np.stack([np.sort(eigvalsh(data_dict[k]['adj_mat'])) for k in keys])

# Sanity anchors on cubic: tr(A^2) = 2|E| = 42 (constant), tr(A^3) = 6*tri.
assert np.allclose((spectra ** 2).sum(axis=1), 42)
assert np.allclose((spectra ** 3).sum(axis=1), 6 * num_triangles)

# Spectral moments tr(A^k)/n, k = 3..8. k=2 excluded: constant on cubic
# (zero variance breaks standardization and carries nothing).
SPEC_KS = range(3, 9)
M = np.stack([(spectra ** k).mean(axis=1) for k in SPEC_KS], axis=1)
# Full sorted spectrum as features: drop the constant Perron eigenvalue (=3).
SPEC_FULL = spectra[:, :-1]
print('moment features:', M.shape, '  full-spectrum features:', SPEC_FULL.shape)

moment features: (509, 6)   full-spectrum features: (509, 13)


In [32]:
# Per-PC decomposition: what explains each embedding axis?
from numpy.linalg import svd

Xc = vectors - vectors.mean(axis=0)
U, S, Vt = svd(Xc, full_matrices=False)
var_ratio = S**2 / (S**2).sum()
pc_scores = Xc @ Vt[:6].T
CYC = np.stack([num_triangles, num_4_cycles, num_5_cycles], axis=1).astype(float)

spec_decomp = {}
print('PC    var%    R2 cycles   R2 moments   R2 full-spec   R2 mom+cyc')
for pc in range(6):
    y = pc_scores[:, pc]
    row = {'var_ratio': var_ratio[pc],
           'r2_cycles':  lstsq_probe(CYC, y).mean(),
           'r2_moments': lstsq_probe(M, y).mean(),
           'r2_spectrum': lstsq_probe(SPEC_FULL, y).mean(),
           'r2_both':    lstsq_probe(np.hstack([M, CYC]), y).mean()}
    spec_decomp[pc + 1] = row
    print(f'PC{pc+1}   {row["var_ratio"]*100:5.1f}   '
          f'{row["r2_cycles"]:+.3f}      {row["r2_moments"]:+.3f}       '
          f'{row["r2_spectrum"]:+.3f}         {row["r2_both"]:+.3f}')

top6 = var_ratio[:6].sum()
for label, col in [('cycles', 'r2_cycles'), ('moments', 'r2_moments'),
                   ('full spectrum', 'r2_spectrum')]:
    share = sum(spec_decomp[i]['var_ratio'] * max(spec_decomp[i][col], 0)
                for i in spec_decomp)
    print(f'\n{label:>14}: {share*100:.1f}% of total embedding variance '
          f'(of {top6*100:.1f}% in top-6 PCs)', end='')
print()

PC    var%    R2 cycles   R2 moments   R2 full-spec   R2 mom+cyc
PC1    51.2   +0.962      +0.962       +0.954         +0.962
PC2    15.3   -0.002      +0.060       +0.195         +0.060
PC3     7.8   +0.128      +0.119       +0.097         +0.119
PC4     7.7   +0.802      +0.811       +0.809         +0.811
PC5     5.3   -0.014      -0.017       +0.010         -0.017
PC6     2.6   -0.033      -0.040       -0.065         -0.040

        cycles: 56.4% of total embedding variance (of 90.0% in top-6 PCs)
       moments: 57.3% of total embedding variance (of 90.0% in top-6 PCs)
 full spectrum: 58.9% of total embedding variance (of 90.0% in top-6 PCs)


In [33]:
# Cospectral mates: zero spectral distance, any embedding separation is
# super-spectral content by construction. Free exhibit if any exist at n=14.
from collections import defaultdict
from itertools import combinations
from scipy.spatial.distance import pdist

spec_hash = [tuple(np.round(s, 8)) for s in spectra]
mates = defaultdict(list)
for k, h in zip(keys, spec_hash):
    mates[h].append(k)
cospectral_groups = [v for v in mates.values() if len(v) > 1]
print(f'{len(cospectral_groups)} cospectral group(s) among {len(keys)} graphs')

if cospectral_groups:
    all_d = pdist(vectors, metric='cityblock')
    cospec_records = []
    for g in cospectral_groups:
        for i, j in combinations(g, 2):
            d = np.abs(vectors[i] - vectors[j]).sum()
            pct = (all_d < d).mean() * 100
            counts_i = (num_triangles[i], num_4_cycles[i], num_5_cycles[i])
            counts_j = (num_triangles[j], num_4_cycles[j], num_5_cycles[j])
            assert counts_i == counts_j, 'cospectral cubic mates must share counts'
            cospec_records.append({'pair': (i, j), 'L1': d, 'percentile': pct,
                                   'counts': counts_i})
            print(f'  pair ({i},{j})  counts {counts_i}  '
                  f'L1 = {d:.4e}  ({pct:.1f} pctile of all pairs)')
    with open('/kaggle/working/n14_e25_spectral.pkl', 'wb') as f:
        pickle.dump({'pc_decomposition': spec_decomp,
                     'cospectral': cospec_records, 'spec_ks': list(SPEC_KS)}, f)
else:
    print('No cospectral cubic mates at n=14 — exhibit moves to the n=16 clone.')
    with open('/kaggle/working/n14_e25_spectral.pkl', 'wb') as f:
        pickle.dump({'pc_decomposition': spec_decomp,
                     'cospectral': [], 'spec_ks': list(SPEC_KS)}, f)

3 cospectral group(s) among 509 graphs
  pair (79,458)  counts (np.int64(2), np.int64(1), np.int64(2))  L1 = 3.9081e-07  (0.1 pctile of all pairs)
  pair (201,203)  counts (np.int64(1), np.int64(3), np.int64(3))  L1 = 1.6919e-05  (5.8 pctile of all pairs)
  pair (234,239)  counts (np.int64(1), np.int64(2), np.int64(4))  L1 = 5.8681e-08  (0.0 pctile of all pairs)


## E2.5 results — the spectral share, and what's left

Spectral-moment content explains 57–59% of total embedding variance
(90% lives in the top six PCs). PC1 (51%) and PC4 (8%) are moment axes.
The cycles and moments+cycles columns coincide by identity — on cubic
graphs, short-cycle counts ARE low spectral moments (tr(A³) = 6·tri,
asserted upstream) — so their agreement is a consistency check, not a
result.

**PC2 (15.3%) is the finding.** The second-largest axis of embedding
variation is essentially unexplained by cycle counts (−0.002), six
spectral moments (0.060), or the full eigenvalue spectrum (0.195). The
embedding varies, at its second-most-dominant scale, along something
non-spectral. Candidate probed below: simple 6-cycles, whose count on
cubic graphs is not spectrally determined.

**Cospectral mates: the "plus," quantified.** Three cospectral pairs
exist in the enumeration; regularity plus cospectrality forces identical
(tri, C4, C5), verified in-cell. All three pairs separate in the
embedding — nonzero L1, orders of magnitude above the float64 noise
floor — at the 0.0, 0.1, and 5.8th percentiles of all pairwise
distances. The super-spectral discriminative residue is present (this is
QCE injectivity, exhibited inside the dataset) and small: 10⁻⁸–10⁻⁵
against typical separations of 10⁻³–10⁻². One exhibit, two paper
sentences: the embedding carries content beyond the spectrum, and the
magnitude of that content is exactly why no efficiency or advantage
claim survives contact with a shot budget.

In [34]:
# C6: the PC2 candidate. Simple 6-cycles are not spectrally determined on
# cubic graphs (closed 6-walks are; simple cycles subtract non-spectral terms).
num_6_cycles = np.array([
    sum(1 for c in nx.simple_cycles(graph(k), length_bound=6) if len(c) == 6)
    for k in keys])
targets['6-cycles'] = num_6_cycles.astype(float)
print(f'C6 range: {num_6_cycles.min()}..{num_6_cycles.max()}')

# Does C6 explain PC2? Alone, and incremental over the six spectral moments.
y = pc_scores[:, 1]
r_c6   = lstsq_probe(num_6_cycles.reshape(-1, 1).astype(float), y).mean()
r_mc6  = lstsq_probe(np.column_stack([M, num_6_cycles]), y).mean()
print(f'PC2 ~ C6 alone: R2 = {r_c6:+.3f}    PC2 ~ moments + C6: R2 = {r_mc6:+.3f}'
      f'    (moments alone: +0.060)')

# And the headline-table row: is C6 coordinate-linear like tri/C4/C5?
sc, al = ridge_probe(vectors, targets['6-cycles'])
print(f'full-vector linear:  6-cycles R2 = {sc.mean():+.3f} +- {sc.std():.3f}   '
      f'{sorted(set(al))}')

with open('/kaggle/working/n14_c6_probe.pkl', 'wb') as f:
    pickle.dump({'num_6_cycles': num_6_cycles, 'pc2_r2_c6': r_c6,
                 'pc2_r2_moments_c6': r_mc6,
                 'e2_row_c6': {'r2_folds': sc, 'alphas': al}}, f)
print('saved n14_c6_probe.pkl')

C6 range: 0..28
PC2 ~ C6 alone: R2 = -0.028    PC2 ~ moments + C6: R2 = +0.210    (moments alone: +0.060)
full-vector linear:  6-cycles R2 = +0.485 +- 0.326   [np.float64(1e-14), np.float64(1e-13), np.float64(1e-12)]
saved n14_c6_probe.pkl


In [36]:
# Cospectral tracer: mate-pair L1 percentile vs truncation depth.
# If percentile rises as k shrinks, the super-spectral residue is
# head-concentrated and the full-vector percentile understates it —
# the signal-concentration function of truncation in QCE, quantified.
MATE_PAIRS = [(79, 458), (201, 203), (234, 239)]
TRUNC_KS = [25, 50, 100, 200, 400, 1000, 4000, vectors.shape[1]]

tracer = {}
print(f'{"k":>6}   ' + '   '.join(f'({i},{j})' for i, j in MATE_PAIRS))
for k in TRUNC_KS:
    Vk = vectors[:, :k]
    all_d = pdist(Vk, metric='cityblock')
    pcts = [(all_d < np.abs(Vk[i] - Vk[j]).sum()).mean() * 100
            for i, j in MATE_PAIRS]
    tracer[k] = pcts
    print(f'{k:>6}   ' + '   '.join(f'{p:7.5f}' for p in pcts))

with open('/kaggle/working/n14_cospectral_tracer.pkl', 'wb') as f:
    pickle.dump({'pairs': MATE_PAIRS, 'ks': TRUNC_KS, 'percentiles': tracer}, f)
print('saved n14_cospectral_tracer.pkl')

     k   (79,458)   (201,203)   (234,239)
    25   0.73094   5.71369   0.19646
    50   0.76265   18.29510   0.10519
   100   0.68762   8.65523   0.04254
   200   0.08508   7.97147   0.03017
   400   0.08740   6.94661   0.00077
  1000   0.06265   5.95579   0.00155
  4000   0.06110   5.81734   0.00000
 16384   0.06110   5.81734   0.00077
saved n14_cospectral_tracer.pkl


In [37]:
# Reference class: count-identical pairs. Separations within a fixed
# (tri, C4, C5) stratum are entirely non-moment fine structure — the
# right yardstick for the cospectral residue. Mates are members of the
# population they're ranked against.
from collections import defaultdict
from itertools import combinations

strata3 = defaultdict(list)
for k in keys:
    strata3[(num_triangles[k], num_4_cycles[k], num_5_cycles[k])].append(k)

ci_pairs = [p for members in strata3.values() if len(members) > 1
            for p in combinations(members, 2)]
n_multi = len([s for s in strata3.values() if len(s) > 1])
print(f'{n_multi} count-identical strata with >1 member, '
      f'{len(ci_pairs)} count-identical pairs '
      f'(of {len(keys) * (len(keys) - 1) // 2} total)')
for i, j in MATE_PAIRS:
    c = (num_triangles[i], num_4_cycles[i], num_5_cycles[i])
    print(f'  mate ({i},{j}): stratum {tuple(int(x) for x in c)}, '
          f'{len(strata3[c])} members, '
          f'{len(strata3[c]) * (len(strata3[c]) - 1) // 2} pairs')

TRUNC = [25, 100, 16384]
ci_result = {}
print(f'\n{"k":>6}   pair          pctile among count-identical pairs')
for k in TRUNC:
    Vk = vectors[:, :k]
    d = np.array([np.abs(Vk[i] - Vk[j]).sum() for i, j in ci_pairs])
    row = {}
    for i, j in MATE_PAIRS:
        dij = np.abs(Vk[i] - Vk[j]).sum()
        row[(i, j)] = (d < dij).mean() * 100
        print(f'{k:>6}   ({i:3d},{j:3d})   {row[(i, j)]:8.3f}')
    ci_result[k] = row

with open('/kaggle/working/n14_count_identical_reference.pkl', 'wb') as f:
    pickle.dump({'strata_sizes': {tuple(int(x) for x in k): len(v)
                                  for k, v in strata3.items()},
                 'n_ci_pairs': len(ci_pairs), 'trunc_ks': TRUNC,
                 'mate_percentiles': ci_result}, f)
print('\nsaved n14_count_identical_reference.pkl')

120 count-identical strata with >1 member, 715 count-identical pairs (of 129286 total)
  mate (79,458): stratum (2, 1, 2), 5 members, 10 pairs
  mate (201,203): stratum (1, 3, 3), 9 members, 36 pairs
  mate (234,239): stratum (1, 2, 4), 7 members, 21 pairs

     k   pair          pctile among count-identical pairs
    25   ( 79,458)     22.937
    25   (201,203)     78.741
    25   (234,239)     13.986
   100   ( 79,458)     24.476
   100   (201,203)     91.888
   100   (234,239)      7.692
 16384   ( 79,458)     10.909
 16384   (201,203)     92.727
 16384   (234,239)      0.000

saved n14_count_identical_reference.pkl


**Reference class: rank the residue against fine structure, not the
moment axis.** Ranked among the 715 count-identical pairs — matched
(tri, C4, C5), so their separations are entirely non-moment fine
structure — the cospectral mates are unremarkable to large: 14th–79th
percentile at k=25, and pair (201,203) sits at the 79th–93rd percentile
at every truncation depth despite zero spectral distance at all orders.
The bottom-percentile global numbers above were the moment axis talking:
count-differing pairs dominate the pooled distribution, and ranking the
residue against them was a category error, corrected here. Within the
fine-structure class, the non-spectral channel carries separations on
the same scale as spectral fine structure — the "plus" is a full
participant in the embedding's fine structure, not a thin layer past
the spectrum. Two caveats stand: the absolute magnitudes (1e-8–1e-5)
sit behind a ~10^10–10^14-shot wall, which is the entire content of
"small" and the load-bearing fact for the no-advantage discipline; and
ranks are against the pooled class, since own-stratum populations
(10–36 pairs) are too thin to rank within.